# Resource-Aware Deep Learning: Hands-On

This notebook will teach you, how to track your own (deep-learning) pipeline's resource consumption using tools like [↗CodeCarbon](https://codecarbon.io/), [↗MLFlow](https://mlflow.org/), and the [↗Lamarr Energy Tracker (LET)](https://github.com/lamarr-institute/lamarr-energy-tracker). This tutorial is built on [↗PyTorch](https://pytorch.org) and [↗PyTorch Lightning](https://lightning.ai/docs/pytorch/stable/). Please make sure you have installed and activated the provided virtual environment (either through `(uv) venv` or `conda/mamba`).

We have provided a small Python package `resource_aware_ml` that implements a number of SRResNet architectures inside the `resource_aware_ml.architectures.model` submodule. If you have not yet installed the package, please do so now inside the repository root directory, e.g. using [uv](https://docs.astral.sh/uv/):

```shell-session
$ uv pip install -e .
```

We can now load the `models` submodule, which implements the following architectures in decreasing complexity:
```
SRResNet18
SRResNet10
SRResNet6
SRResNet4
```
![SRResNet Overview](assets/resnet_sc.png)

In [1]:
from rich import print  # nicer prints

from resource_aware_ml.architectures import models

Choose one of the architectures. Beware that higher complexity will mean more GPU memory consumption. Make sure **not** to initialise the model. This will be handled by the `TrainModule` below.

In [41]:
# Choose your architecture here
model = models.SRResNet4

print(model())

SRResNet4(
  (preBlock): Sequential(
    (0): Conv2d(2, 64, kernel_size=(9, 9), stride=(1, 1), padding=(4, 4), groups=2)
    (1): PReLU(num_parameters=1)
  )
  (final): Sequential(
    (0): Conv2d(64, 2, kernel_size=(9, 9), stride=(1, 1), padding=(4, 4), groups=2)
    (1): PReLU(num_parameters=1)
  )
  (blocks): Sequential(
    (0): SRBlock(
      (idconv): Identity()
      (pool): Identity()
      (convs): Sequential(
        (0): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False, padding_mode=reflect)
        (1): Dropout(p=False, inplace=False)
        (2): InstanceNorm2d(64, eps=1e-05, momentum=0.1, affine=False, bias=False, track_running_stats=False)
        (3): PReLU(num_parameters=1)
        (4): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False, padding_mode=reflect)
        (5): Dropout(p=False, inplace=False)
        (6): InstanceNorm2d(64, eps=1e-05, momentum=0.1, affine=False, bias=False, track_running_stats=False)
      )
    )
  )
)

Before we go over the training loop, we have to import and initialize a training module. The training module sets up the training, validation, and inference steps of the model, as well as the optimizer. During the training and validation steps it also calls the loss function `loss_fn` that is used to monitor the training process.

In [42]:
import torch
torch.cuda.is_available()

True

In [43]:
from torch.nn import L1Loss
from torch.optim import AdamW

from resource_aware_ml.training.trainer import TrainModule

In [44]:
# Change the Optimizer or loss function here.
train_config = dict(model=model, loss_fn=L1Loss, optimizer=AdamW, lr=1e-3)

train_module = TrainModule(train_config)

We will also need a data module that handles the data loading for us. The data module implemented in the package is designed to load the dataset that is provided in the repository. The data module inherits from the [↗`lightning.LightningDataModule`](https://lightning.ai/docs/pytorch/stable/api/lightning.pytorch.core.LightningDataModule.html) and is simply a collection of dataloaders for the training, validation, test, and inference stages.

In [45]:
from resource_aware_ml.io.data import H5DataModule

In [46]:
data_module = H5DataModule(
    data_dir="./data",  # Path to the data in the repository relative to this notebook
    batch_size=256,  # Change if you run out of memory
    fourier=True,  # Our data is in Fourier space
    num_workers=16,  # Change the number of concurrent CPU cores loading the data
)

## Task 1: Logging

Implement a logger that logs the experiment data such as the train loss and validation loss to **MLFlow**. Have a look at the lightning docs for more information on [loggers](https://lightning.ai/docs/pytorch/stable/api_references.html#loggers). Save the logger in a variable `logger`.

In [47]:
# Implement your solution here
from pathlib import Path

from lightning.pytorch.loggers import CSVLogger, MLFlowLogger

mlruns_uri = "sqlite:///" + str(Path("./build/mlruns.db").expanduser().resolve())
logger = [
    CSVLogger(save_dir="./build", name="training_logs"),
    MLFlowLogger(tracking_uri=mlruns_uri),
]

Now we can set up the [↗`lightning.Trainer`](https://lightning.ai/docs/pytorch/stable/api/lightning.pytorch.trainer.trainer.Trainer.html). Here, we can set

In [48]:
from lightning import Trainer
from lightning.pytorch.callbacks import RichProgressBar

In [49]:
trainer = Trainer(
    max_epochs=10,
    devices="auto",
    accelerator="auto",  # Change to gpu, or cpu, if you like
    precision="32-true",
    logger=logger,
    callbacks=RichProgressBar(),
    log_every_n_steps=20,  # set to batch size
    strategy="auto",
)

Trainer will use only 1 of 4 GPUs because it is running inside an interactive / notebook environment. You may try to set `Trainer(devices=4)` but please note that multi-GPU inside interactive / notebook environments is considered experimental and unstable. Your mileage may vary.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


## Task 2: Training & Emission Tracking

To start the training process, we will have to use the `Trainer.fit` method. In order to track the emissions, we will need to use CodeCarbon or the Lamarr Energy Tracker (LET).

In [50]:
import numpy as np
from lamarr_energy_tracker import EnergyTracker

from resource_aware_ml.utils import get_predictions

## 

Your task is to log the following metrics and parameters using MLFlow and CodeCarbon/LET:
<table>
<tr><td>
    
| Metric                         | Variable Name              |
| ------------------------------ | -------------------------- |
| Number of trainable parameters | `num_trainable_parameters` |
| Total runtime                  | `running_time_total`       |
| Runtime per epoch              | `running_time`             |
| Total power draw               | `power_draw_total`         |
| Power draw per epoch           | `power_draw`               |
| Mean Total Flux Ratio          | `mean_total_flux`          |
| Mean Peak Flux Ratio           | `mean_peak_flux`           |
| Epochs trained                 | `epochs`                   |

</td><td>

| Parameter                | Variable Name  |
| ------------------------ | -------------- |
| Model name               | `model`        |
| Dataset                  | `dataset`      |
| Architecture (GPU Model) | `architecture` |
| Task (`training`)        | `task`         |

</td></tr> </table>

To get the predictions use the function
```python
resource_aware_ml.utils.get_predictions(trainer, task=Literal["training" | "inference"]) -> tuple[np.ndarray, np.ndarray] | np.ndarray
```
from the `resource_aware_ml` package. For the `training` task, this function returns predictions and targets.
The mean total flux ratio and the mean peak flux ratios can then be calculated as

$$
    R = \texttt{mean}\left(\frac{\texttt{preds}}{\texttt{targets}}\right).
$$

Values close to `1.0` would indicate good predictions. In order to use the ratios as suitable metrics for STREP, minimize the ratios
as
```python
mean_<metric>_flux = abs(1.0 - <metric>)
```

***Hint***: Use the `.sum()` and `.max()` methods to get the total and peak fluxes from the data.

In [51]:
emissions_path = Path("./build").expanduser().resolve()

# Track the training loop
with EnergyTracker(project_name="dpg_tutorial", output_dir=emissions_path) as tracker:
    trainer.fit(model=train_module, datamodule=data_module)

# When using multiple loggers, get the experiment and
# run id from the MLFLowLogger instance
mlflow_logger = next(
    logger for logger in trainer.loggers if isinstance(logger, MLFlowLogger)
)
experiment = mlflow_logger.experiment
run_id = mlflow_logger._run_id

# Get total number of samples (train + valid)
num_samples = trainer.datamodule.train_length + trainer.datamodule.valid_length

# Get predictions and targets
preds, targets = get_predictions(trainer, task="training")
mean_total_flux = np.abs(1.0 - np.mean(preds.sum() / targets.sum()))
mean_peak_flux = np.abs(1.0 - np.mean(preds.max() / targets.max()))

# get results from tracker
last_results = tracker.results.iloc[-1]

metrics = dict(
    running_time_total=last_results["duration"],
    running_time=last_results["duration"] / num_samples,
    power_draw_total=last_results["energy_consumed"],
    power_draw=last_results["energy_consumed"] / num_samples,
    mean_total_flux=mean_total_flux,
    mean_peak_flux=mean_peak_flux,
    epoch=trainer.current_epoch,
)

for key, val in metrics.items():
    experiment.log_metric(
        key=key,
        value=val,
        run_id=run_id,
    )

model_name = trainer.model.model.__class__.__name__
model_name += "_" + trainer.model.optimizer.__name__
model_name += "_" + trainer.model.loss_fn.__class__.__name__

params = dict(
    model=model_name,
    dataset="Radio Simulation HDF5",
    architecture=last_results["gpu_model"],
    task="train",
)

for key, val in params.items():
    experiment.log_param(
        key=key,
        value=val,
        run_id=run_id,
    )

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3]


┏━━━┳━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name    ┃ Type      ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model   │ SRResNet4 │ 84.2 K │ train │     0 │
│ 1 │ loss_fn │ L1Loss    │      0 │ train │     0 │
└───┴─────────┴───────────┴────────┴───────┴───────┘

Trainable params: 84.2 K                                                                                           
Non-trainable params: 0                                                                                            
Total params: 84.2 K                                                                                               
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 20                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/scratch/aknierim/envs/resource-awareness/lib/python3.12/site-packages/lightning/pytorch/utilities/_pytree.py:21: 
`isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` 
instead.

/scratch/aknierim/envs/resource-awareness/lib/python3.12/site-packages/lightning/pytorch/loops/fit_loop.py:317: The
number of training batches (1) is smaller than the logging interval Trainer(log_every_n_steps=20). Set a lower 
value for log_every_n_steps if you want to see logs for the training epoch.

`Trainer.fit` stopped: `max_epochs=10` reached.



Tracker stopped - this experiment consumed 1.136 Wh.


Using CodeCarbon 3.2.3, the energy consumption of running all experiments on an AMD EPYC 9555 64-Core Processor and 4 x NVIDIA RTX PRO 6000 Blackwell Server Edition is estimated to 1.136 Wh.This corresponds to estimated carbon emissions of 0.432 gCO2-equivalents, assuming a carbon intensity of 380 gCO2/kWh~\cite{lamarr_energy_tracker,codecarbon}. Note that these numbers are underestimations of actual resource consumption and do not account for overhead factors or embodied impacts~\cite{ai_energy_validation}.

For comparison, this is equivalent to 1 rolls of recycled toilet paper. Comparisons are based on ``How bad are bananas? The Carbon Footprint of everything'' by Mike Berners-Lee. Greystone Books 2011.




## Task 3: Inference Emission Tracking

Your task is to log the following metrics and parameters using MLFlow and CodeCarbon/LET for inference:
<table>
<tr><td>
    
| Metric                         | Variable Name              |
| ------------------------------ | -------------------------- |
| Number of trainable parameters | `num_trainable_parameters` |
| Total runtime                  | `running_time_total`       |
| Runtime per epoch              | `running_time`             |
| Total power draw               | `power_draw_total`         |
| Power draw per epoch           | `power_draw`               |
| Mean Total Flux Ratio          | `mean_total_flux`          |
| Mean Peak Flux Ratio           | `mean_peak_flux`           |
| Epochs trained                 | `epochs`                   |

</td><td>

| Parameter                | Variable Name  |
| ------------------------ | -------------- |
| Model name               | `model`        |
| Dataset                  | `dataset`      |
| Architecture (GPU Model) | `architecture` |
| Task (`training`)        | `task`         |


</td></tr> </table>

In [52]:
from natsort import natsorted

versions = natsorted(Path("./build/training_logs/").glob("version_*"))
latest_ckpt = natsorted((versions[-1] / "checkpoints").glob("epoch*.ckpt"))[-1]

In [53]:
# Change checkpoint path accordingly
train_module = TrainModule.load_from_checkpoint(
    latest_ckpt,
    weights_only=False,
)

# Reinitialize MLFlowLogger and Trainer, otherwise we will get an error
logger = [
    CSVLogger(save_dir="./build", name="inference_logs"),
    MLFlowLogger(tracking_uri=mlruns_uri),
]
trainer = Trainer(
    max_epochs=10,
    accelerator="auto",  # Change to gpu, or cpu, if you like
    precision="32-true",
    logger=logger,
    callbacks=RichProgressBar(),
    log_every_n_steps=20,  # set to batch size
)

emissions_path = Path("./build").expanduser().resolve()

# Track the training loop
with EnergyTracker(project_name="dpg_tutorial", output_dir=emissions_path) as tracker:
    trainer.predict(model=train_module, datamodule=data_module)

# When using multiple loggers, get the experiment and
# run id from the MLFLowLogger instance
mlflow_logger = next(
    logger for logger in trainer.loggers if isinstance(logger, MLFlowLogger)
)
experiment = mlflow_logger.experiment
run_id = mlflow_logger._run_id

num_samples = trainer.datamodule.predict_length

# Get predictions and targets
# NOTE: We're using the 'testing' mode here since we have a
# testing dataset with ground truths. In true inference
# We could not compute the metrics below.
preds, targets = get_predictions(trainer, task="testing")
mean_total_flux = np.abs(1.0 - np.mean(preds.sum() / targets.sum()))
mean_peak_flux = np.abs(1.0 - np.mean(preds.max() / targets.max()))

# Get results from tracker
last_results = tracker.results.iloc[-1]

metrics = dict(
    running_time_total=last_results["duration"],
    running_time=last_results["duration"] / num_samples,
    power_draw_total=last_results["energy_consumed"],
    power_draw=last_results["energy_consumed"] / num_samples,
    mean_total_flux=mean_total_flux,
    mean_peak_flux=mean_peak_flux,
    train_loss=1,  # Since we are writing to the same file as
    val_loss=1,  # the training run, we have to log both losses.
    epoch=trainer.current_epoch,  # In this case, we also have to log the epochs
)

for key, val in metrics.items():
    experiment.log_metric(
        key=key,
        value=val,
        run_id=run_id,
    )

model_name = trainer.model.model.__class__.__name__
model_name += "_" + trainer.model.optimizer.__name__
model_name += "_" + trainer.model.loss_fn.__class__.__name__

params = dict(
    model=model_name,
    dataset="Radio Simulation HDF5",
    architecture=last_results["gpu_model"],
    task="inference",
)

for key, val in params.items():
    experiment.log_param(
        key=key,
        value=val,
        run_id=run_id,
    )

Trainer will use only 1 of 4 GPUs because it is running inside an interactive / notebook environment. You may try to set `Trainer(devices=4)` but please note that multi-GPU inside interactive / notebook environments is considered experimental and unstable. Your mileage may vary.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3]
/scratch/aknierim/envs/resource-awareness/lib/python3.12/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is depreca

Output()


Tracker stopped - this experiment consumed 0.114 Wh.


Using CodeCarbon 3.2.3, the energy consumption of running all experiments on an AMD EPYC 9555 64-Core Processor and 4 x NVIDIA RTX PRO 6000 Blackwell Server Edition is estimated to 1.250 Wh.This corresponds to estimated carbon emissions of 0.475 gCO2-equivalents, assuming a carbon intensity of 380 gCO2/kWh~\cite{lamarr_energy_tracker,codecarbon}. Note that these numbers are underestimations of actual resource consumption and do not account for overhead factors or embodied impacts~\cite{ai_energy_validation}.

For comparison, this is equivalent to 5.9 imported bananas. Comparisons are based on ``How bad are bananas? The Carbon Footprint of everything'' by Mike Berners-Lee. Greystone Books 2011.




## Task 4: Train More Models

Repeat tasks 2 and 3 for different model configurations, e.g. by changing the architecture, the optimizer, or the loss function.

## Task 5: MLFlow Export

Export the logged data to a `.csv` file using MLFlow's CLI tool.

## Task 6: Visualisation Using STREP

Visualise the data using the STREP framework.